# Finding the Critical Temperature

Running the simulation, magnetisation and energy data were provided throughout the evolution of the system. This data had to carefully analysed to determine the "shape" of the distribution of each. 

First data must be pulled from the simulations, and then can be unpacked. This is done so via the `np.load` (and previously `np.save`) function, to efficiently store data. 

The data is unpacked into dictionaries for easy access and readability. Here, for the magnetisation, we store the mean magnetisation, the susceptibility (i.e. the variance of the magnetisation) and the Binder cumulant. For energy, we store the mean, the specific heat (i.e. the variance of the energy) and the Binder cumulant. 

Currently, the "warmup" period (i.e. the period for the magnetisation/energy to stabilise, or get to a resonable state) is arbitrarily set at 1500 steps. This should be modified later to vary depending on the size and temperature of each individual system.

Below is an example for magnetisation.

In [ ]:
warmup = 1500

m = mag[warmup:]
            

total_length = len(m)
sub_sample_length = total_length//nsplits
_mean,_chi,_kurt = [],[],[]

for i in range(nsplits):
    _m = m[i*sub_sample_length:(i+1)*sub_sample_length:freq]   
    _mean.append( _m.mean()/N**2 )           
    _chi.append( _m.var()/T/N**2)

    binder = 1- np.mean(_m**4)/3./np.mean(_m**2)**2
    _kurt.append( binder) 

    mag_dict [(N,T)] = {'mag_per_particle':(np.mean(_mean),np.std(_mean)),
                        'chi_per_particle':(np.mean(_chi),np.std(_chi)),
                        'kurtosis': (np.mean(_kurt),np.std(_kurt))
                                           }

`mag_dict` stores all the information related to the magnetic field. `n_splits` determines how many times the magnetisation/energy is  "subsample", in order to estimate the error, with frequency determining how many points to select from the range. 

With this data, graphs of Binder Cumulant/Kurtosis against temperature can be plotted. As mentioned in *Theory.Binder Cumulant*, at the critical temperature, the kurtosis of each system size should be the same - *scale-invariance* is observed, due to the correlation length tending to infinity at this point. 

We can fit polynomials to the data to better see the intersection, taking care to ensure the critical temperature is near the midpoint of the temperature domain, such that the polynomial fit is accurate. 

![Fig. 1: A plot of Binder Cumulant against temperature for 6 systems of different size, with T between 2 & 2.5.](/images/mag_k-t_graph.png)

This is reasonably close to the analytical value of $T_c = 2.269\text{K}$ [ref needed]. However, the fitting is inconsistent due to the nature of polynomial fitting. This can especially be seen when using larger sets of points (see Fig. 2).

![Fig. 2: Fitting with a larger number of plots. The range of values for the intersection is difficult, and anomalous fitting artifacts cause the critical temperature to stray from the true "intersection".](/images/bad_fitting.png)

There are a few ways to potentially counteract this. One could simply limit the range of points that classify as "intersections", however this is not a general solution (this is done in Fig. 1). By lowering the degree of the polynomial fit, this could be improved, but the larger system sizes still caused issues with anomalous intersection points. 

![Fig. 3: Fitting with low degree polynomials. Large system sizes cause anomalous intersections to appear, skewing the critical temperature reading.](/images/bad_fitting_2.png)

A better approach to fitting was to use `numpy.spline` to fit a low-order polynomial between each pair of data points as shown in Fig. 4 (completeley eliminating anomalous intersections), and then use a least-variance fit to determine the point of intersection i.e. the point at which all the lines are closet to each other (Figs. 4,6).
 
![Fig. 4: Using spline fit to fit the data magnetisation data.](/images/spline_mag_good.png)

![Fig. 5: A closeup of the spline fitting for the data. Some fitting artifacts remain, however the critical point is clearly visible.](/images/spline_mag_zoomed.png)

![Fig. 6: Finding the minimal variance between spline fits for each different system size. The peak demonstrates the closest value - i.e. the intersection point. This finds a critical temperature of 2.269..., within 5 decimal places of the analytical value. ](/images/reciprocal_variance.png)

The bootstrap method can be used to estimate the error here, by running the fitting algorithm many times, with points randomly shifted according to their error. 